# Shop Manager — Build Android APK on Google Colab

**How to use this notebook:**
1. Zip your `shop_mobile` folder on your PC into `shop_mobile.zip`.
2. Run the cells below **in order**, top to bottom (click the ▶ button on each, or Runtime → Run all).
3. Cell 2 will ask you to upload `shop_mobile.zip` — pick the file from your PC.
4. The build takes 20–45 minutes on the *first* run (downloading the Android SDK/NDK). Keep this browser tab open.
5. The last cell gives you a direct download link for the finished `.apk`.

If Colab disconnects you (long idle, or free-tier timeout), just re-open the notebook and run the cells again from the top — nothing is saved between sessions on the free tier.

In [ ]:
# CELL 1 — Install build tools (takes a few minutes)
!apt update -qq
!apt install -y -qq build-essential git python3-venv libffi-dev libssl-dev \
    zlib1g-dev libsqlite3-dev openjdk-17-jdk unzip
!pip install --quiet buildozer==1.5.0 cython==0.29.33
print("Build tools installed.")

In [ ]:
# CELL 2 — Upload your shop_mobile.zip
from google.colab import files
print("Choose your shop_mobile.zip file...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f"Uploaded: {zip_name}")

In [ ]:
# CELL 3 — Unzip the project
import zipfile, os

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/build')

# Find the folder that actually contains buildozer.spec (handles both
# "shop_mobile.zip contains shop_mobile/..." and "...contains files directly").
project_dir = None
for root, dirs, filenames in os.walk('/content/build'):
    if 'buildozer.spec' in filenames:
        project_dir = root
        break

if project_dir is None:
    raise SystemExit("Could not find buildozer.spec inside the zip. Make sure you zipped the shop_mobile folder itself (with buildozer.spec at its top level).")

print(f"Project found at: {project_dir}")
%cd {project_dir}
!ls

In [ ]:
# CELL 4 — Build the APK (this is the long one — 20-45 min first time)
!yes | buildozer android debug

In [ ]:
# CELL 5 — Find and download the finished APK
import glob
from google.colab import files

apks = glob.glob('bin/*.apk')
if not apks:
    print("No APK found — scroll up through Cell 4's output to see the error.")
    print("Copy the last ~30 lines of red/error text and send them back for a fix.")
else:
    apk_path = apks[0]
    print(f"Found: {apk_path}")
    files.download(apk_path)
    print("Download started — check your browser's downloads.")

## Next: install it on your phone

1. Transfer the downloaded `.apk` to your Android phone (Google Drive, email to yourself, USB cable — whatever's easiest).
2. Open it from your phone's Files app / Downloads.
3. Android will warn about "install from unknown sources" the first time — this is expected since it's not from the Play Store. Allow it for this file.
4. The app installs like any other and appears on your home screen / app drawer as **Shop Manager**.

## If Cell 4 fails

Buildozer's error output is usually specific about what went wrong (a missing package, a version conflict, etc.). Scroll up to find the actual error (not just the last line), copy the last 20-30 lines, and send them back — Android build failures are almost always fixable once we can see the real error.